# 03 · Results — where does the model go wrong?

Accuracy alone hides the interesting part. This notebook shows:

* the learning curves
* per-class accuracy and the most common confusions
* the test images the model got wrong

Needs a trained model in `models/` and the test set in `data/split/test`.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision import datasets, models, transforms

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL = ROOT / "models" / "dish_classifier.pt"
classes = json.loads(MODEL.with_suffix(".classes.json").read_text())

tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
model = models.mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, len(classes))
model.load_state_dict(torch.load(MODEL, map_location="cpu"))
model.eval()
test = datasets.ImageFolder(ROOT / "data" / "split" / "test", tf)
assert test.classes == classes
print(len(test), "test images,", len(classes), "classes")

## Learning curves

In [ ]:
hist = json.loads((ROOT / "models" / "history.json").read_text())
ep = [h["epoch"] for h in hist]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.5))
a.plot(ep, [h["train_loss"] for h in hist], label="train"); a.plot(ep, [h["val_loss"] for h in hist], label="val")
a.set_title("loss"); a.legend()
b.plot(ep, [h["train_acc"] for h in hist], label="train"); b.plot(ep, [h["val_acc"] for h in hist], label="val")
b.set_title("accuracy"); b.legend()
plt.show()

## Predictions on the test set

In [ ]:
rows = []
with torch.no_grad():
    for i, (x, y) in enumerate(test):
        probs = torch.softmax(model(x.unsqueeze(0)), 1)[0]
        conf, pred = probs.max(0)
        rows.append({"path": test.samples[i][0], "true": classes[y], "pred": classes[pred], "conf": float(conf)})

correct = sum(r["true"] == r["pred"] for r in rows)
print(f"Test accuracy: {correct / len(rows):.2%}  ({correct}/{len(rows)})\n")
for c in classes:
    rs = [r for r in rows if r["true"] == c]
    ok = sum(r["pred"] == c for r in rs)
    print(f"{c:15s} {ok / max(len(rs), 1):6.1%}  ({ok}/{len(rs)})")

## Most common confusions

In [ ]:
pairs = Counter((r["true"], r["pred"]) for r in rows if r["true"] != r["pred"])
for (t, p), n in pairs.most_common(10):
    print(f"{t:15s} -> {p:15s} {n}x")

## Images the model got wrong

In [ ]:
wrong = [r for r in rows if r["true"] != r["pred"]]
cols = 5
n = len(wrong)
if n == 0:
    print("No mistakes on the test set.")
else:
    fig, axes = plt.subplots((n + cols - 1) // cols, cols, figsize=(cols * 3, ((n + cols - 1) // cols) * 3.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, r in zip(axes.flat, wrong):
        ax.imshow(Image.open(r["path"]).convert("RGB"))
        ax.set_title(f"true: {r['true']}\npred: {r['pred']} ({r['conf']:.0%})", fontsize=9, color="#C0392B")
    plt.tight_layout()
    plt.show()

## Notes

* With only a handful of test images per class, one mistake moves a class by 20–25 %. Collect more data before drawing conclusions.
* Dishes with the same red gochujang sauce (tteokbokki, dakgalbi, jeyuk bokkeum) are the hardest to tell apart — add more varied photos of them.